## 1. Setup

### Libraries

In [1]:
# weather data 
import meteostat as ms
from meteostat import Stations

# from src
from src.spark_config import initialize_spark # spark session
from src.helper_functions import get_coordinates, get_paths 

# spark biblioteker fra labene
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first, input_file_name,
    regexp_extract, min, max, year
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

# other libraries
import pandas as pd
import numpy as np
import regex as re
from datetime import datetime, timedelta

# specify java version
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

### Initialize

In [2]:
# set spark session
spark = initialize_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/26 15:15:24 WARN Utils: Your hostname, LaptopEHM, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/26 15:15:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 15:15:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.2.1
Application ID: local-1795702526325
Spark UI: http://10.255.255.254:4040


### Define Data Paths

In [3]:
# paths for statsbomb data
paths = get_paths()

## 2. Matches

### Load and Filter La Liga Matches

In [4]:
matches_df = spark.read.parquet("data/bronze/statsbomb/matches")
laliga_matches = matches_df.filter(
    col("competition.competition_name") == "La Liga")

### Data Quality Before Cleaning

In [5]:
print(f"Number of matches in La Liga (before cleaning): ", laliga_matches.count())
print(f"Number of unique matches in La Liga (before cleaning): ", laliga_matches.select("match_id").distinct().count())

Number of matches in La Liga (before cleaning):  867
Number of unique matches in La Liga (before cleaning):  867


In [6]:
# data quality of matches before cleaning
invalid_matches_before = laliga_matches.filter(
    col("match_id").isNull() |
    col("kick_off").isNull() |
    col("stadium.name").isNull() |
    col("season.season_name").isNull() |
    col("match_date").isNull()
).count()

invalid_scores_before = laliga_matches.filter(
    (col("home_score") < 0) |
    (col("away_score") < 0)).count()

print(f"Invalid matches (before cleaning): ", invalid_matches_before)
print(f"Invalid scores (before cleaning): ", invalid_scores_before)

Invalid matches (before cleaning):  0
Invalid scores (before cleaning):  0


### Transform and Clean Matches

In [7]:

# forslag til første utdrag av data
silver_matches = laliga_matches.select(
        col("match_id"),
        col("kick_off"),
        col("stadium.name").alias("stadium"),
        col("home_team.home_team_name").alias("home_team"),
        col("away_team.away_team_name").alias("away_team"),
        col("home_team.home_team_id").alias("home_team_id"),
        col("away_team.away_team_id").alias("away_team_id"),
        col("home_score"), # sånn at vi kan se på om hjemmelaget spiller bedre i ulike værforhold?
        col("away_score"),
        col("season.season_name").alias("season"),
        to_date("match_date").alias("match_date")
        )

silver_matches.show(5)
silver_matches.printSchema()

+--------+------------+--------------------+--------------------+--------------+------------+------------+----------+----------+---------+----------+
|match_id|    kick_off|             stadium|           home_team|     away_team|home_team_id|away_team_id|home_score|away_score|   season|match_date|
+--------+------------+--------------------+--------------------+--------------+------------+------------+----------+----------+---------+----------+
| 3825739|17:00:00.000|            Bernabéu|         Real Madrid|Sporting Gijón|         220|        1041|         5|         1|2015/2016|2016-01-17|
| 3825848|20:00:00.000|Estadio Ciudad de...|          Levante UD|         Eibar|         221|         322|         2|         2|2015/2016|2015-09-23|
| 3825895|22:00:00.000|Estadio de Gran C...|          Las Palmas|       Sevilla|         208|         213|         2|         0|2015/2016|2015-09-23|
| 3825894|18:15:00.000|Estadio Abanca-Ri...|RC Deportivo La C...|        Getafe|         219|       

### Data Quality After Cleaning

In [8]:
print(f"Number of matches in La Liga (after cleaning): ", silver_matches.count())
print(f"Number of unique matches in La Liga (after cleaning): ", silver_matches.select("match_id").distinct().count())

Number of matches in La Liga (after cleaning):  867
Number of unique matches in La Liga (after cleaning):  867


In [9]:
invalid_matches_after = silver_matches.filter(
    col("match_id").isNull() |
    col("kick_off").isNull() |
    col("stadium").isNull() |
    col("season").isNull() |
    col("match_date").isNull()
).count()

invalid_scores_after = silver_matches.filter(
    (col("home_score") < 0) |
    (col("away_score") < 0)).count()

print(f"Invalid matches (after cleaning): ", invalid_matches_after)
print(f"Invalid scores (after cleaning): ", invalid_scores_after)

Invalid matches (after cleaning):  0
Invalid scores (after cleaning):  0


In [10]:
matches_df.printSchema()

root
 |-- away_score: long (nullable = true)
 |-- away_team: struct (nullable = true)
 |    |-- away_team_gender: string (nullable = true)
 |    |-- away_team_group: string (nullable = true)
 |    |-- away_team_id: long (nullable = true)
 |    |-- away_team_name: string (nullable = true)
 |    |-- country: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- managers: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- country: struct (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- dob: string (nullable = true)
 |    |    |    |-- id: long (nullable = true)
 |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- nickname: string (nullable = true)
 |-- competition: struct (nullable = true)
 |    |-- competition_id: long (nullable = true)
 |    |-- competition_nam

## 3. Events

### Load and Filter La Liga Events

In [11]:
events_df = spark.read.parquet("data/bronze/statsbomb/events")

laliga_events = events_df.join(
    laliga_matches.select("match_id"),
    on="match_id",
    how="inner"
)

### Data Quality Before Cleaning

In [12]:
print(f"Number of events in La Liga (before cleaning): ", laliga_events.count())
print(f"Number of unique events in La Liga (before cleaning): ", laliga_events.select("id").distinct().count())

Number of events in La Liga (before cleaning):  3132716


Number of unique events in La Liga (before cleaning):  3132716


In [13]:
invalid_events_before = laliga_events.filter(
    col("match_id").isNull() |
    col("id").isNull() |
    (col("type.name") == "Pass") & col("pass").isNull()
).count()

print(f"Invalid events (before cleaning): ", invalid_events_before)

Invalid events (before cleaning):  0


### Transform and Clean Events

In [14]:
silver_events = laliga_events.select(
    col("match_id"),
    col("id").alias("event_id"),
    col("type.name").alias("event_type"),
    col("bad_behaviour"),
    col("pass"),
    col("miscontrol"),
    col("team.name").alias("team"),
    col("player.name").alias("player"),
    col("shot"),
    col("foul_committed")
    )

26/11/26 15:16:11 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
{"ts": "2026-11-26 15:16:11.911", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `bad_behaviour` cannot be resolved. Did you mean one of the following? [`duration`, `ball_receipt`, `ball_recovery`, `location`, `period`]. SQLSTATE: 42703", "context": {"file": "line 5 in cell [14]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o189.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `bad_behaviour` cannot be resolved. Did you mean one of the following? [`duration`, `ball_receipt`, `ball_recovery

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `bad_behaviour` cannot be resolved. Did you mean one of the following? [`duration`, `ball_receipt`, `ball_recovery`, `location`, `period`]. SQLSTATE: 42703;
'Project [match_id#414L, id#391 AS event_id#609, type#412.name AS event_type#610, 'bad_behaviour, pass#398, 'miscontrol, team#410.name AS team#611, player#401.name AS player#612, shot#407, foul_committed#388]
+- Project [match_id#414L, 50_50#378, ball_receipt#379, ball_recovery#380, block#381, carry#382, clearance#383, counterpress#384, dribble#385, duel#386, duration#387, foul_committed#388, foul_won#389, goalkeeper#390, id#391, index#392L, interception#393, location#394, minute#395L, off_camera#396, out#397, pass#398, period#399L, play_pattern#400, player#401, ... 14 more fields]
   +- Join Inner, (match_id#414L = match_id#10L)
      :- Relation [50_50#378,ball_receipt#379,ball_recovery#380,block#381,carry#382,clearance#383,

In [ ]:
silver_events.printSchema()
silver_events.show(5)

### Data Quality After Cleaning

In [ ]:
print(f"Number of events in La Liga (after cleaning): ", silver_events.count())
print(f"Number of unique events in La Liga (after cleaning): ", silver_events.select("event_id").distinct().count())

In [ ]:
invalid_events_after = silver_events.filter(
    col("match_id").isNull() |
    col("event_id").isNull() |
    (col("event_type") == "Pass") & col("pass").isNull()
).count()

print(f"Invalid events (after cleaning): ", invalid_events_after)

In [ ]:
events_df.printSchema()

## 4. Weather 

In [ ]:
# get number of matches in each season
silver_matches.groupBy("season").count().orderBy("season").show()

In [ ]:
# get number of matches by each year
silver_matches.groupBy(
    year("match_date").alias("year")
).count().orderBy("year").show(truncate=False)

In [ ]:
# stadiums we need to find weather stations for
silver_matches.select("stadium").distinct().orderBy("stadium").show(50, truncate=False)

### Load Weather Data

### Data Quality Before Cleaning

### Transform and Clean Weather Data

### Data Quality After Cleaning

### Join Weather With Matches

## 5. Write Silver Data To Parquet

In [ ]:
from src.helper_functions import get_coordinates

# Known stadium
print(get_coordinates("Bernabéu", "Spain"))

# Unknown stadium
print(get_coordinates("Lyse Arena", "Norway"))

# Non-existent stadium
print(get_coordinates("Nonexistent Stadium XYZ123"))

In [ ]:
from geopy.geocoders import Nominatim

geo_locator = Nominatim(user_agent="dat535-football-stats")

location = geo_locator.geocode(
    "Stamford Bridge, England",
    addressdetails=True
)

print(location.raw)

{'place_id': 280836130, 'licence': 'Data © OpenStreetMap contributors, ODbL 1.0. http://osm.org/copyright', 'osm_type': 'way', 'osm_id': 5986169, 'lat': '51.4816869', 'lon': '-0.1910336', 'class': 'leisure', 'type': 'stadium', 'place_rank': 30, 'importance': 0.6028355034306307, 'addresstype': 'leisure', 'name': 'Stamford Bridge', 'display_name': 'Stamford Bridge, Britannia Gate, Walham Green, Fulham, London Borough of Hammersmith and Fulham, Greater London, England, SW6 1HD, United Kingdom', 'address': {'leisure': 'Stamford Bridge', 'road': 'Britannia Gate', 'neighbourhood': 'Walham Green', 'suburb': 'Fulham', 'city_district': 'London Borough of Hammersmith and Fulham', 'ISO3166-2-lvl8': 'GB-HMF', 'city': 'Greater London', 'state': 'England', 'ISO3166-2-lvl4': 'GB-ENG', 'postcode': 'SW6 1HD', 'country': 'United Kingdom', 'country_code': 'gb'}, 'boundingbox': ['51.4807699', '51.4826215', '-0.1924468', '-0.1896121']}


26/11/26 17:30:18 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 3273117 ms exceeds timeout 120000 ms
26/11/26 17:30:18 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/26 17:30:25 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:70